In [3]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
import re

In [ ]:
data = pd.read_csv("Data/bbc_news.csv" ,index_col=0)
data.info()
data.head(1)

In [ ]:
data['desc_proc'] = data['description'].apply(lambda x : [word.lower() for word in x.split()] )

In [ ]:
en_words = stopwords.words('english')
data['desc_proc'] = data['desc_proc'].apply(lambda x : [''.join(word) for word in x if word not in en_words])

In [ ]:
data['desc_proc'] = data['desc_proc'].apply(lambda x : [re.sub(r'[^\s\w]', '', word) for word in x])
data['desc_proc'] = data['desc_proc'].apply(lambda x : ' '.join(x))

In [ ]:
data['desc_proc_tokens'] = data.apply(lambda x : word_tokenize(x['desc_proc']),axis=1)

In [ ]:
lm = WordNetLemmatizer()
data['desc_proc_tokens'] = data['desc_proc_tokens'].apply(lambda tokens : [lm.lemmatize(token) for token in tokens])
clean_token = sum(data['desc_proc_tokens'] ,[])
unigram = pd.DataFrame(nltk.ngrams(clean_token ,1)).value_counts()
bigram = pd.DataFrame(nltk.ngrams(clean_token ,2)).value_counts()
unigram.head()
print(bigram[:10])

In [ ]:
import spacy
from spacy import displacy
from spacy import tokenizer
nlp = spacy.load('en_core_web_sm')
data_clean = data.drop(['guid', 'link'] ,axis=1)

In [ ]:
text = ' '.join([word for word in clean_token])
spacy_text = nlp(text)
pos_rows = [{"Token": token.text, "POS": token.pos_} for token in spacy_text]
df_pos = pd.DataFrame(pos_rows)


In [ ]:
df_pos_counts = (df_pos.value_counts(["Token", "POS"]).reset_index(name="Counts"))
nouns = (df_pos_counts[df_pos_counts.POS == "NOUN"].sort_values("Counts", ascending=False).head(10))

In [ ]:
ner_rows = [{"Token": ent.text, "NER": ent.label_} for ent in spacy_text.ents]
df_ner = pd.DataFrame(ner_rows)
displacy.render(spacy_text[:100], style='ent', jupyter=True)

In [ ]:
df_ner_counts = (df_ner.value_counts(["Token", "NER"]).reset_index(name="Counts").sort_values("Counts", ascending=False))
df_ner_label_counts = df_ner["NER"].value_counts()


In [ ]:
print("\nTop Named Entities:")
print(df_ner_counts.head())

print("\nEntity Type Distribution:")
print(df_ner_label_counts.head(10))

print("\nMost Frequent Nouns:")
print(nouns)